# Monocular Depth & Geometry Estimation Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Depth metrics

In [ ]:
```python

import torch

def abs_rel_error(pred, target, mask=None):

    if mask is not None:

        pred = pred[mask]

        target = target[mask]

    return (torch.abs(pred - target) / target.clamp(min=1e-6)).mean().item()

def delta_accuracy(pred, target, threshold=1.25, mask=None):

    if mask is not None:

        pred = pred[mask]

        target = target[mask]

    ratio = torch.maximum(pred / target.clamp(min=1e-6), target / pred.clamp(min=1e-6))

    return (ratio < threshold).float().mean().item()

In [ ]:
```

Always mask invalid depth pixels (zero, NaN, saturated) before evaluation.

### Step 2: Scale-and-shift alignment

For relative-depth models, align prediction to ground truth before computing metrics. Least-squares fit of `a * pred + b = target`:

In [ ]:
```python

def align_scale_shift(pred, target, mask=None):

    if mask is not None:

        p = pred[mask]

        t = target[mask]

    else:

        p = pred.flatten()

        t = target.flatten()

    A = torch.stack([p, torch.ones_like(p)], dim=1)

    coeffs, *_ = torch.linalg.lstsq(A, t.unsqueeze(-1))

    a, b = coeffs[:2, 0]

    return a * pred + b

In [ ]:
```

Run `align_scale_shift` before `abs_rel_error` when evaluating MiDaS / Depth Anything.

### Step 3: Lift depth to a point cloud

In [ ]:
```python

import numpy as np

def depth_to_point_cloud(depth, intrinsics):

    H, W = depth.shape

    fx, fy, cx, cy = intrinsics

    v, u = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")

    z = depth

    x = (u - cx) * z / fx

    y = (v - cy) * z / fy

    return np.stack([x, y, z], axis=-1)

depth = np.random.uniform(0.5, 4.0, (240, 320))

intr = (320.0, 320.0, 160.0, 120.0)

pc = depth_to_point_cloud(depth, intr)

print(f"point cloud shape: {pc.shape}  (H, W, 3)")

In [ ]:
```

One function, every 3D-lifted application. Export the point cloud to `.ply` and open in MeshLab or CloudCompare.

### Step 4: Smoke test with a synthetic depth scene

In [ ]:
```python

def synthetic_depth(size=96):

    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")

    # Floor: linear gradient from near (top) to far (bottom)

    depth = 1.0 + (yy / size) * 4.0

    # Box in the middle: closer

    mask = (np.abs(xx - size / 2) < size / 6) & (np.abs(yy - size * 0.6) < size / 6)

    depth[mask] = 2.0

    return depth.astype(np.float32)

gt = torch.from_numpy(synthetic_depth(96))

pred = gt + 0.3 * torch.randn_like(gt)  # simulated prediction

aligned = align_scale_shift(pred, gt)

print(f"before align  absRel = {abs_rel_error(pred, gt):.3f}")

print(f"after align   absRel = {abs_rel_error(aligned, gt):.3f}")

In [ ]:
```

### Step 5: Depth Anything V3 usage (reference)

In [ ]:
```python

import torch

from transformers import pipeline

from PIL import Image

pipe = pipeline(task="depth-estimation", model="LiheYoung/depth-anything-v2-large")

image = Image.open("street.jpg").convert("RGB")

out = pipe(image)

depth_np = np.array(out["depth"])

In [ ]:
```

Three lines. `out["depth"]` is a PIL grayscale; convert to numpy for math. For Depth Anything V3 specifically, swap the model id once released; the API is unchanged.

## Exercises

In [ ]:
1. **(Easy)** Run Depth Anything V2 on any 10 images of your desk. Save depth as grayscale PNGs and inspect. Identify one object whose predicted depth looks wrong and explain why the monocular cues failed.
2. **(Medium)** Given RGB + depth from Depth Anything V2, lift to a point cloud and render with `open3d`. Compare two scenes (indoor / outdoor) and note which looks more believable.
3. **(Hard)** Take five pairs of images that differ only by a known object's position (e.g. bottle moved 30 cm closer). Use UniDepth to predict metric depth on both. Report the predicted distance delta vs the true 30 cm.